# EI-Beginner 任务三

这个 notebook 对应 README 中的第三个任务：

1. 复现模仿学习经典 baseline `Diffusion Policy`
2. 同时学习 HuggingFace 机器人学习框架 `LeRobot`

这里不直接从大规模真实机器人数据集起步，而是先搭一个“最小可运行”的模仿学习实验：

- 用程序合成一个 2D reaching / grasping 风格的专家数据集
- 先做最基础的 Behavior Cloning (BC)
- 再实现一个简化版 Diffusion Policy，学习动作 chunk 的条件生成
- 最后把思路映射到 LeRobot 的真实工作流


## 任务要求拆解

任务三真正考察的不是“把官方仓库跑起来”这么单一，而是下面四件事是否理解清楚：

- 为什么模仿学习能用于机械臂抓取
- 为什么 Diffusion Policy 比逐步回归单步动作更稳健
- 数据集里 observation / action / episode 的组织方式是什么
- 如何把 toy problem 迁移到 LeRobot + 真实机器人数据集

因此这个 notebook 分成四部分：

- Part A: 构造一个专家数据集
- Part B: 用 BC 训练动作预测器
- Part C: 用简化版 Diffusion Policy 训练动作 chunk 生成器
- Part D: 对接 LeRobot 的数据格式、训练命令和后续扩展


In [ ]:
# 如果环境缺依赖，可以取消注释后运行
# %pip install torch matplotlib numpy


In [ ]:
import math
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE


## Part A: 构造一个最小专家数据集

为了把重点放在“模仿学习”本身，我们先不用真实机械臂仿真器，而是构造一个二维平面里的抓取近似任务。

每个 episode 里：

- 机械臂末端当前位置是 `(x, y)`
- 目标物体位置是 `(gx, gy)`
- 抓手开合状态是 `gripper`
- 专家策略会先移动到目标附近，再闭合夹爪

我们把一个时刻的 observation 定义为：

`obs = [ee_x, ee_y, goal_x, goal_y, gripper]`

把一个时刻的 action 定义为：

`act = [dx, dy, d_gripper]`


In [ ]:
@dataclass
class Episode:
    observations: np.ndarray   # [T, obs_dim]
    actions: np.ndarray        # [T, act_dim]


def expert_policy_step(state, goal, step_size=0.08, close_threshold=0.06):
    pos = state[:2]
    gripper = state[2]
    delta = goal - pos
    dist = np.linalg.norm(delta)

    if dist > close_threshold:
        move = delta / (dist + 1e-8) * min(step_size, dist)
        grip_action = 0.0
    else:
        move = np.zeros(2, dtype=np.float32)
        grip_action = 1.0 - gripper

    return np.array([move[0], move[1], grip_action], dtype=np.float32)


def rollout_expert_episode(horizon=32):
    ee = np.random.uniform(-1.0, 1.0, size=2).astype(np.float32)
    goal = np.random.uniform(-0.8, 0.8, size=2).astype(np.float32)
    gripper = np.array([0.0], dtype=np.float32)

    obs_list, act_list = [], []
    for _ in range(horizon):
        obs = np.concatenate([ee, goal, gripper]).astype(np.float32)
        act = expert_policy_step(np.concatenate([ee, gripper]), goal)
        ee = ee + act[:2]
        gripper = np.array([np.clip(gripper[0] + act[2], 0.0, 1.0)], dtype=np.float32)
        obs_list.append(obs)
        act_list.append(act)

    return Episode(np.stack(obs_list), np.stack(act_list))


def build_dataset(num_episodes=512, horizon=32):
    return [rollout_expert_episode(horizon=horizon) for _ in range(num_episodes)]


episodes = build_dataset(num_episodes=512, horizon=32)
len(episodes), episodes[0].observations.shape, episodes[0].actions.shape


In [ ]:
# 可视化一个专家轨迹
example = episodes[0]
obs = example.observations
acts = example.actions
positions = obs[:, :2]
goal = obs[0, 2:4]

plt.figure(figsize=(5, 5))
plt.plot(positions[:, 0], positions[:, 1], 'o-', label='end-effector path')
plt.scatter(goal[0], goal[1], c='red', s=120, marker='*', label='goal')
plt.scatter(positions[0, 0], positions[0, 1], c='green', s=80, label='start')
plt.axis('equal')
plt.grid(True)
plt.legend()
plt.title('Expert trajectory')
plt.show()

print('last 5 gripper actions:', acts[-5:, 2])


### 这里你应该理解什么

- 模仿学习最核心的输入不是 reward，而是专家 demonstration
- demonstration 的基本单位通常是 trajectory / episode
- 对抓取任务，action 往往不止包含末端位移，还包含夹爪控制
- 如果 observation 设计得不合理，后面再强的模型也学不好


## Part B: Behavior Cloning 基线

Behavior Cloning 可以理解为“监督学习版控制策略”：

- 输入 observation
- 输出 action
- 直接最小化 `pred_action` 和 `expert_action` 的误差

它简单、直接，但容易出现 covariate shift：

- 训练时看到的是专家状态分布
- 测试时一旦自己走偏，就会进入没见过的状态


In [ ]:
class BCDataset(Dataset):
    def __init__(self, episodes):
        self.obs = np.concatenate([ep.observations for ep in episodes], axis=0).astype(np.float32)
        self.act = np.concatenate([ep.actions for ep in episodes], axis=0).astype(np.float32)

    def __len__(self):
        return len(self.obs)

    def __getitem__(self, idx):
        return torch.from_numpy(self.obs[idx]), torch.from_numpy(self.act[idx])


class BCPolicy(nn.Module):
    def __init__(self, obs_dim=5, act_dim=3, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, act_dim),
        )

    def forward(self, obs):
        return self.net(obs)


bc_dataset = BCDataset(episodes)
bc_loader = DataLoader(bc_dataset, batch_size=128, shuffle=True)

bc_policy = BCPolicy().to(DEVICE)
optimizer = torch.optim.Adam(bc_policy.parameters(), lr=1e-3)

bc_losses = []
for epoch in range(80):
    running = 0.0
    for batch_obs, batch_act in bc_loader:
        batch_obs = batch_obs.to(DEVICE)
        batch_act = batch_act.to(DEVICE)
        pred = bc_policy(batch_obs)
        loss = F.mse_loss(pred, batch_act)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += loss.item()
    bc_losses.append(running / len(bc_loader))

plt.figure(figsize=(6, 3))
plt.plot(bc_losses)
plt.title('BC training loss')
plt.xlabel('epoch')
plt.ylabel('MSE')
plt.grid(True)
plt.show()


In [ ]:
@torch.no_grad()
def rollout_bc(policy, horizon=32):
    ee = np.random.uniform(-1.0, 1.0, size=2).astype(np.float32)
    goal = np.random.uniform(-0.8, 0.8, size=2).astype(np.float32)
    gripper = np.array([0.0], dtype=np.float32)

    traj = []
    for _ in range(horizon):
        obs = np.concatenate([ee, goal, gripper]).astype(np.float32)
        obs_t = torch.from_numpy(obs).unsqueeze(0).to(DEVICE)
        act = policy(obs_t).squeeze(0).cpu().numpy()
        ee = ee + np.clip(act[:2], -0.1, 0.1)
        gripper = np.array([np.clip(gripper[0] + act[2], 0.0, 1.0)], dtype=np.float32)
        traj.append((obs.copy(), act.copy()))

    traj = list(traj)
    positions = np.stack([x[0][:2] for x in traj])
    goal = traj[0][0][2:4]
    return positions, goal, traj


positions, goal, traj = rollout_bc(bc_policy)
plt.figure(figsize=(5, 5))
plt.plot(positions[:, 0], positions[:, 1], 'o-', label='BC rollout')
plt.scatter(goal[0], goal[1], c='red', s=120, marker='*', label='goal')
plt.axis('equal')
plt.grid(True)
plt.legend()
plt.show()

print('final position:', positions[-1])
print('goal:', goal)
print('final gripper estimate:', traj[-1][0][-1] + traj[-1][1][-1])


### 为什么 BC 不够

对于机械臂抓取，单步 action regression 往往有几个问题：

- 动作分布可能是多峰的，MSE 容易学成“平均动作”
- 单步预测缺少对未来动作序列结构的建模
- 一旦偏离专家轨迹，误差会逐步累积

Diffusion Policy 的关键改进是：

- 不只预测单步 action，而是预测 action chunk
- 通过去噪生成来拟合复杂、多峰的动作分布


## Part C: 简化版 Diffusion Policy

下面不是完整论文实现，而是一个足够说明问题的最小版本：

- 条件输入：当前 observation
- 预测目标：未来 `H` 步动作 chunk
- 训练方式：给动作 chunk 加噪，学习预测噪声
- 推理方式：从高斯噪声开始迭代去噪，得到一个动作序列

这和论文/开源实现相比省略了图像编码器、时间窗口条件、UNet backbone 等复杂部分，但保留了 diffusion policy 的核心思想。


In [ ]:
CHUNK_HORIZON = 8
OBS_DIM = 5
ACT_DIM = 3

class DiffusionChunkDataset(Dataset):
    def __init__(self, episodes, chunk_horizon=8):
        self.samples = []
        for ep in episodes:
            T = len(ep.observations)
            for t in range(T - chunk_horizon + 1):
                obs = ep.observations[t]
                chunk = ep.actions[t:t + chunk_horizon]
                self.samples.append((obs.astype(np.float32), chunk.astype(np.float32)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        obs, chunk = self.samples[idx]
        return torch.from_numpy(obs), torch.from_numpy(chunk)


chunk_dataset = DiffusionChunkDataset(episodes, chunk_horizon=CHUNK_HORIZON)
chunk_loader = DataLoader(chunk_dataset, batch_size=128, shuffle=True)
len(chunk_dataset), chunk_dataset[0][0].shape, chunk_dataset[0][1].shape


In [ ]:
class TinyTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(
            torch.arange(half, device=t.device, dtype=torch.float32)
            * (-math.log(10000.0) / max(half - 1, 1))
        )
        angles = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([torch.sin(angles), torch.cos(angles)], dim=1)
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0, 1))
        return emb


class TinyDiffusionPolicy(nn.Module):
    def __init__(self, obs_dim=5, act_dim=3, horizon=8, hidden_dim=256, time_dim=64):
        super().__init__()
        self.horizon = horizon
        self.act_dim = act_dim
        self.time_embed = TinyTimeEmbedding(time_dim)
        in_dim = obs_dim + horizon * act_dim + time_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, horizon * act_dim),
        )

    def forward(self, obs, noisy_chunk, t):
        b = obs.shape[0]
        flat_chunk = noisy_chunk.reshape(b, -1)
        time_feat = self.time_embed(t)
        x = torch.cat([obs, flat_chunk, time_feat], dim=1)
        pred_noise = self.net(x)
        return pred_noise.reshape(b, self.horizon, self.act_dim)


In [ ]:
TIMESTEPS = 50
betas = torch.linspace(1e-4, 0.02, TIMESTEPS, device=DEVICE)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)


def q_sample(x0, t, noise):
    a_bar = alpha_bars[t].view(-1, 1, 1)
    return torch.sqrt(a_bar) * x0 + torch.sqrt(1.0 - a_bar) * noise


def diffusion_loss(model, obs, chunk):
    b = obs.shape[0]
    t = torch.randint(0, TIMESTEPS, (b,), device=obs.device)
    noise = torch.randn_like(chunk)
    noisy_chunk = q_sample(chunk, t, noise)
    pred_noise = model(obs, noisy_chunk, t)
    return F.mse_loss(pred_noise, noise)


In [ ]:
diffusion_policy = TinyDiffusionPolicy(obs_dim=OBS_DIM, act_dim=ACT_DIM, horizon=CHUNK_HORIZON).to(DEVICE)
optimizer = torch.optim.Adam(diffusion_policy.parameters(), lr=1e-3)

diff_losses = []
for epoch in range(120):
    running = 0.0
    for obs_batch, chunk_batch in chunk_loader:
        obs_batch = obs_batch.to(DEVICE)
        chunk_batch = chunk_batch.to(DEVICE)
        loss = diffusion_loss(diffusion_policy, obs_batch, chunk_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += loss.item()
    diff_losses.append(running / len(chunk_loader))

plt.figure(figsize=(6, 3))
plt.plot(diff_losses)
plt.title('Diffusion policy training loss')
plt.xlabel('epoch')
plt.ylabel('noise prediction MSE')
plt.grid(True)
plt.show()


In [ ]:
@torch.no_grad()
def sample_action_chunk(model, obs, clip_value=1.0):
    obs = obs.to(DEVICE)
    x = torch.randn(obs.shape[0], CHUNK_HORIZON, ACT_DIM, device=DEVICE)

    for step in reversed(range(TIMESTEPS)):
        t = torch.full((obs.shape[0],), step, device=DEVICE, dtype=torch.long)
        pred_noise = model(obs, x, t)
        alpha = alphas[step]
        alpha_bar = alpha_bars[step]
        beta = betas[step]
        x = (x - (beta / torch.sqrt(1 - alpha_bar)) * pred_noise) / torch.sqrt(alpha)
        if step > 0:
            x = x + torch.sqrt(beta) * torch.randn_like(x)
        x = torch.clamp(x, -clip_value, clip_value)
    return x


@torch.no_grad()
def rollout_diffusion(policy, horizon=32):
    ee = np.random.uniform(-1.0, 1.0, size=2).astype(np.float32)
    goal = np.random.uniform(-0.8, 0.8, size=2).astype(np.float32)
    gripper = np.array([0.0], dtype=np.float32)

    positions = []
    actions = []
    steps = 0
    while steps < horizon:
        obs = np.concatenate([ee, goal, gripper]).astype(np.float32)
        obs_t = torch.from_numpy(obs).unsqueeze(0).to(DEVICE)
        chunk = sample_action_chunk(policy, obs_t).squeeze(0).cpu().numpy()

        for action in chunk:
            ee = ee + np.clip(action[:2], -0.1, 0.1)
            gripper = np.array([np.clip(gripper[0] + action[2], 0.0, 1.0)], dtype=np.float32)
            positions.append(ee.copy())
            actions.append(action.copy())
            steps += 1
            if steps >= horizon:
                break

    return np.stack(positions), goal, np.stack(actions)


positions, goal, actions = rollout_diffusion(diffusion_policy)
plt.figure(figsize=(5, 5))
plt.plot(positions[:, 0], positions[:, 1], 'o-', label='Diffusion rollout')
plt.scatter(goal[0], goal[1], c='red', s=120, marker='*', label='goal')
plt.axis('equal')
plt.grid(True)
plt.legend()
plt.show()

print('final position:', positions[-1])
print('goal:', goal)
print('last 5 gripper actions:', actions[-5:, 2])


### Diffusion Policy 的最小理解框架

你至少应该能回答下面几个问题：

- 为什么这里训练的是“预测噪声”而不是直接预测动作
- 为什么 policy 输出一个 action chunk，而不是只输出一步
- 为什么 condition 可以是当前 observation，也可以扩展成历史图像 + 力觉 + proprioception
- 为什么 diffusion model 更适合拟合复杂、非线性的多模态动作分布

如果这些问题还回答不清，说明还停留在“会跑代码”而不是“理解方法”。


## Part D: LeRobot 工作流

LeRobot 是 HuggingFace 面向机器人学习的统一框架。你应该重点关注三件事：

- 数据集格式如何统一
- policy / env / dataset / evaluation 如何解耦
- 如何复用社区已有数据集和 baseline

下面给一个建议学习路径。


In [ ]:
# 可选：安装 LeRobot
# 说明：LeRobot 依赖较多，这里默认不自动执行。
# 如果你准备认真复现，建议单独创建干净环境。

# %pip install lerobot


### 1. 先理解 LeRobot 的数据长什么样

典型机器人数据集会包含：

- `observation.images.*` 或相机图像
- `observation.state` 或关节/末端状态
- `action`
- `episode_index`, `frame_index`, `timestamp`

和本 notebook 的 toy data 对照起来，其实本质没变：

- 这里只是把 observation 从 5 维向量换成了多模态输入
- 把 action 从 3 维末端控制换成真实机械臂控制量
- 把 episode 管理做得更标准化


In [ ]:
# 这里给出一个“把 toy episode 转成更标准记录”的示例
records = []
for ep_idx, ep in enumerate(episodes[:3]):
    for t, (obs, act) in enumerate(zip(ep.observations, ep.actions)):
        records.append({
            'episode_index': ep_idx,
            'frame_index': t,
            'observation.state': obs.tolist(),
            'action': act.tolist(),
        })

records[:3]


### 2. 用 LeRobot 复现 baseline 的实际步骤

建议按这个顺序推进：

1. 先跑通一个官方示例数据集和 checkpoint
2. 再训练一个最小 policy，确认数据加载、日志和评估都正常
3. 最后再替换成你自己的抓取数据集

如果你直接跳到“自己采数据 + 自己改模型”，很容易卡在工程细节里。


In [ ]:
# 下面这些命令是“学习路径模板”，不是要求你在本 notebook 里全部执行。
# 不同版本的 LeRobot CLI 可能会调整，使用前请以官方文档/README 为准。

lerobot_commands = [
    'git clone https://github.com/huggingface/lerobot.git',
    'cd lerobot',
    'pip install -e .',
    'python -m lerobot.scripts.visualize_dataset --help',
    'python -m lerobot.scripts.train --help',
]

for cmd in lerobot_commands:
    print(cmd)


### 3. 你最终要能迁移到什么层级

完成这个 notebook 后，建议继续做下面三步：

- 在仿真环境里采集你自己的 pick-and-place demonstration
- 把 observation 扩展成图像 + 低维状态
- 用 LeRobot 的 dataset/policy pipeline 复现一次官方 baseline

如果你能走完这三步，就不只是“看过 Diffusion Policy”，而是真正碰过 imitation learning 的核心链路。


## 交付建议

如果你要把任务三整理成可以提交的作业，建议包含：

- 你对 Diffusion Policy 的方法总结
- 你自己跑出来的训练曲线和 rollout 可视化
- 你对 BC 与 Diffusion Policy 差异的分析
- 你对 LeRobot 数据组织与训练流程的总结
- 你下一步打算如何把 toy demo 换成真实机械臂/仿真抓取数据
